# Step 4+5: baseline inference, prompts v1/v2/v3 (Exp 1, 2, 3)

Loads base Qwen2.5-Coder-1.5B-Instruct **once** and runs it on all 1,034 dev prompts for each of v1, v2, v3 in turn.

- Model is public: no HF token needed.
- **No vLLM.** An earlier run showed `pip install vllm` replaces Kaggle's preinstalled PyTorch with a different CUDA build than Kaggle's preinstalled torchaudio, which crashes on import. This notebook uses Kaggle's **preinstalled** torch + transformers only, and never installs anything that could pull in a different torch/torchvision/torchaudio.
- The first cell prints Python/torch/CUDA/transformers versions and the GPU name, and stops immediately with a clear error if no GPU is present.
- Batched HF `generate`, fp16, left-padded, greedy decoding, `max_new_tokens=256`, stops at the model's real end-of-turn token (`<|im_end|>`, not just the tokenizer's default EOS). Prompts are sorted by length before batching (so a batch isn't padded out to one very long outlier) and restored to original order before saving. Batch size halves automatically and retries on a CUDA out-of-memory error instead of crashing the run.
- Input files are found by **recursively searching `/kaggle/input`** for the expected filenames — never a fixed dataset slug — and the notebook fails with a clear message if any is missing.
- **Resumable:** outputs are flushed to disk every 32 examples. A version whose output file already has all 1,034 rows is skipped entirely; a partial file is continued from where it stopped.
- Output: `/kaggle/working/predictions_v1.jsonl`, `predictions_v2.jsonl`, `predictions_v3.jsonl`, and `run_metadata.json`.

**Settings before Run All:** Settings → Accelerator: GPU T4 x2, Internet: On. (Only one T4 is used — intentional.) Add Input → your uploaded prompts dataset. No secrets needed.

## How to resume an interrupted run
1. Make sure the run you want to resume from was saved with **Save Version → Save & Run All** (an interactive-only run's outputs aren't attachable later).
2. Open this notebook again (new session is fine).
3. **Add Input** (right panel) → switch to the **Notebook Output Files** tab (not Datasets) → search for this notebook by name → pick the specific earlier version you want to resume from → **Add**.
4. Keep the original prompts dataset attached too (it should already be there).
5. **Run All** again. The notebook detects `predictions_v*.jsonl` files from that earlier version under `/kaggle/input/...`, copies them into `/kaggle/working`, prints `RESUMING`, and continues each version from the first row that hasn't been generated yet — skipping any version that's already fully done.


In [ ]:
# Environment check -- catch a broken environment before spending any GPU time.
import sys
print("Python:", sys.version)

import torch
print("torch:", torch.__version__, "| torch's CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available in this session. Check Settings -> Accelerator is set to GPU T4 x2, "
        "then Run All again."
    )

print("GPU:", torch.cuda.get_device_name(0))

# only install transformers if it's actually missing, and never in a way that could
# pull in a different torch/torchvision/torchaudio build (see PROGRESS.md decision on
# the vLLM failure this project hit)
try:
    import transformers
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "transformers"], check=True)
    import transformers

print("transformers:", transformers.__version__)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single T4 on purpose

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SEED = 0
MAX_NEW_TOKENS = 256
MAX_MODEL_LEN = 4096      # covers our longest v3 prompt (~3528 tok) + generation headroom
CHUNK_SIZE = 32            # examples per flush/resume checkpoint
INITIAL_BATCH_SIZE = 8     # halves automatically and retries on CUDA OOM

OUT_DIR = "/kaggle/working"
VERSIONS = ["v1", "v2", "v3"]

In [ ]:
import glob

def find_input_file(filename):
    """Recursively search /kaggle/input for `filename`. Returns the first match or None."""
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return matches[0] if matches else None

PROMPT_PATHS = {}
missing = []
for v in VERSIONS:
    fname = f"prompts_{v}_dev.jsonl"
    path = find_input_file(fname)
    if path is None:
        missing.append(fname)
    else:
        PROMPT_PATHS[v] = path
        print(f"found {fname} -> {path}")

if missing:
    raise FileNotFoundError(
        "Missing required input file(s) under /kaggle/input: " + ", ".join(missing) +
        ". Make sure the prompts dataset (prompts_v1_dev.jsonl, prompts_v2_dev.jsonl, "
        "prompts_v3_dev.jsonl) is attached as a notebook input."
    )

In [ ]:
import shutil

print("Checking for a previous run to resume from...")
resumed_any = False
for v in VERSIONS:
    fname = f"predictions_{v}.jsonl"
    working_path = os.path.join(OUT_DIR, fname)
    if os.path.exists(working_path):
        continue  # already present this session (e.g. re-running a cell) -- leave it alone
    prev_path = find_input_file(fname)
    if prev_path is not None:
        shutil.copy(prev_path, working_path)
        print(f"  resuming {v}: copied previous output {prev_path} -> {working_path}")
        resumed_any = True

if resumed_any:
    print(">>> RESUMING from a previous run's attached output.")
else:
    print(">>> STARTING FRESH -- no previous run output found under /kaggle/input.")

In [ ]:
import json

def load_prompts(version):
    rows = []
    with open(PROMPT_PATHS[version], encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def load_done(path):
    """Already-written rows from a possibly-partial predictions file, keyed by idx.
    Drops a truncated trailing line (e.g. from a mid-write crash) instead of erroring on it."""
    done = {}
    if not os.path.exists(path):
        return done
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            done[row["idx"]] = row
    return done

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for correct batched causal-LM generation

# Qwen's chat template ends a turn with <|im_end|>, which may not be the tokenizer's
# default eos_token_id -- stop generation at both, to actually stop where the turn ends.
eos_ids = {tokenizer.eos_token_id}
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
if im_end_id is not None and im_end_id != getattr(tokenizer, "unk_token_id", None):
    eos_ids.add(im_end_id)
eos_ids = sorted(eos_ids)
print("stopping generation at token ids:", eos_ids, [tokenizer.decode([i]) for i in eos_ids])

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to("cuda")
model.eval()
print("model loaded on", next(model.parameters()).device)

In [ ]:
def generate_batch_safe(prompts, batch_size):
    """Generate for `prompts`, halving batch_size and retrying on a CUDA OOM instead of crashing."""
    try:
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                        max_length=MAX_MODEL_LEN).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                  eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id)
        gen_only = out[:, enc["input_ids"].shape[1]:]
        return tokenizer.batch_decode(gen_only, skip_special_tokens=True)
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        torch.cuda.empty_cache()
        if batch_size <= 1:
            raise
        half = batch_size // 2
        print(f"  CUDA OOM at batch_size={batch_size}, halving to {half} and retrying")
        results = []
        for i in range(0, len(prompts), half):
            results.extend(generate_batch_safe(prompts[i:i + half], half))
        return results


def generate_chunk(rows, batch_size):
    """rows: list of {idx, db_id, question, prompt}. Returns raw output strings in the
    SAME order as `rows` -- sorts by prompt length internally so a batch isn't padded out
    to one long outlier, then restores original order before returning."""
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]["prompt"]))
    sorted_prompts = [rows[i]["prompt"] for i in order]
    sorted_outputs = []
    for i in range(0, len(sorted_prompts), batch_size):
        batch = sorted_prompts[i:i + batch_size]
        sorted_outputs.extend(generate_batch_safe(batch, batch_size))
    outputs = [None] * len(rows)
    for pos, orig_idx in enumerate(order):
        outputs[orig_idx] = sorted_outputs[pos]
    return outputs

In [ ]:
import time

run_metadata = {
    "model": MODEL_NAME, "backend": "transformers batched generate (no vLLM)", "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy", "eos_token_ids": eos_ids,
    "python_version": sys.version, "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda, "transformers_version": transformers.__version__,
    "gpu_name": torch.cuda.get_device_name(0), "versions": {},
}

for version in VERSIONS:
    rows = load_prompts(version)
    out_path = os.path.join(OUT_DIR, f"predictions_{version}.jsonl")
    done = load_done(out_path)

    if len(done) >= len(rows):
        print(f"{version}: already complete ({len(done)}/{len(rows)}) -- skipping")
        run_metadata["versions"][version] = {"n_examples": len(rows), "status": "already complete, skipped"}
        continue

    # rewrite the file with just the valid completed rows (drops any corrupt tail from a crash),
    # then continue appending the rest
    with open(out_path, "w", encoding="utf-8") as f:
        for idx in sorted(done):
            f.write(json.dumps(done[idx]) + "\n")

    remaining = [r for r in rows if r["idx"] not in done]
    if done:
        print(f"{version}: RESUMING -- {len(done)}/{len(rows)} already done, {len(remaining)} remaining")
    else:
        print(f"{version}: STARTING FRESH -- {len(remaining)} to do")

    version_start = time.time()
    with open(out_path, "a", encoding="utf-8") as f:
        for i in range(0, len(remaining), CHUNK_SIZE):
            chunk = remaining[i:i + CHUNK_SIZE]
            t0 = time.time()
            raw_outputs = generate_chunk(chunk, INITIAL_BATCH_SIZE)
            elapsed = time.time() - t0
            per_example_time = elapsed / len(chunk)
            for r, raw in zip(chunk, raw_outputs):
                f.write(json.dumps({
                    "idx": r["idx"], "db_id": r["db_id"], "question": r["question"],
                    "raw_output": raw, "gen_time_sec": per_example_time,
                }) + "\n")
            f.flush()
            os.fsync(f.fileno())
            done_so_far = len(done) + i + len(chunk)
            print(f"{version}: {done_so_far}/{len(rows)} done, last chunk {elapsed:.1f}s")
    version_elapsed = time.time() - version_start
    run_metadata["versions"][version] = {
        "n_examples": len(rows), "newly_generated": len(remaining), "total_time_sec": version_elapsed,
    }
    print(f"=== {version} done in {version_elapsed:.1f}s, wrote {out_path} ===")

with open(os.path.join(OUT_DIR, "run_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)
print(json.dumps(run_metadata, indent=2))

## Download these files from /kaggle/working/ when done:
- `predictions_v1.jsonl`
- `predictions_v2.jsonl`
- `predictions_v3.jsonl`
- `run_metadata.json`

If you need to stop partway through, that's fine — just use **Save Version → Save & Run All** before stopping, so the partial outputs are attachable later (see "How to resume" above).